In [ ]:
import os
import sys
import pandas as pd
from src.utils.db_utils import get_connection, execute_query

def byeweeks():
    """Bye weeks """
    query = """
    select *
    from stats.byeweek
    where season = '2024
    """
    return  execute_query(query)

bye_weekdf = byeweeks()

prtint(bye_weekdf)

def get_teamstats_with_game_context(gamesummaryid=None, season=None):
    """Get teamstats with game context fields populated"""
    
    # Base query with the CASE logic
    query = """
    SELECT 
        ts.season,
        ts.week,
        ts.gamesummaryid,
        ts.teamid,
        ts.hometeamid,
        ts.awayteamid,
        ts.total_yards,
        -- Game context fields based on home/away logic
        CASE 
            WHEN ts.teamid = gs.hometeamid THEN gs.days_since_last_game_hometeam
            WHEN ts.teamid = gs.awayteamid THEN gs.days_since_last_game_awayteam
            ELSE NULL
        END as days_since_last_game,
        CASE 
            WHEN ts.teamid = gs.hometeamid THEN gs.hometeam_weeks_from_bye
            WHEN ts.teamid = gs.awayteamid THEN gs.awayteam_weeks_from_bye
            ELSE NULL
        END as weeks_from_bye
    FROM stats.teamstats ts
    LEFT JOIN stats.gamesummary gs ON ts.gamesummaryid = gs.gamesummaryid
    """
    
    # Add WHERE conditions based on parameters
    where_conditions = []
    if gamesummaryid:
        where_conditions.append(f"ts.gamesummaryid = '{gamesummaryid}'")
    if season:
        where_conditions.append(f"ts.season = {season}")
    
    if where_conditions:
        query += " WHERE " + " AND ".join(where_conditions)
    
    query += " ORDER BY ts.gamesummaryid, ts.teamid"
    
    return execute_query(query)

df_test = get_teamstats_with_game_context(gamesummaryid='gs-202410DETHOU')
#print("Test game results:")
#print(df_test)


Successfully connected to the database!
Test game results:
   season  week    gamesummaryid teamid hometeamid awayteamid  total_yards  \
0    2024    10  gs-202410DETHOU    DET        HOU        DET          345   
1    2024    10  gs-202410DETHOU    HOU        HOU        DET          248   

   days_since_last_game  weeks_from_bye  
0                     7              -5  
1                    10               4  
